In [1]:
import os
import numpy as np
import open3d as o3d
import cv2

# ——— Utility functions ———

def load_velodyne_bin(path):
    """Load a KITTI .bin as Nx4 float32, return Nx3 XYZ."""
    data = np.fromfile(path, dtype=np.float32)
    pts = data.reshape(-1, 4)[:, :3]
    return pts

# def read_calib(calib_path):
#     """
#     Parse a KITTI calib file of the form written by save_kitti_calib:
#     - Lines P0: …  P1: …  P2: …  P3: …
#     - R0_rect:
#     - Tr_velo_to_cam:
#     """
#     P = {}
#     R0 = None
#     Tr = None
#     with open(calib_path, 'r') as f:
#         for line in f:
#             toks = line.strip().split()
#             if toks[0].startswith('P'):
#                 cam_id = int(toks[0][1])
#                 vals = list(map(float, toks[1:]))
#                 P[cam_id] = np.array(vals).reshape(3,4)
#             elif toks[0].startswith('R0_rect'):
#                 vals = list(map(float, toks[1:]))
#                 R0 = np.array(vals).reshape(3,3)
#             elif toks[0].startswith('Tr_velo_to_cam'):
#                 vals = list(map(float, toks[1:]))
#                 Tr = np.array(vals).reshape(3,4)
#     return P, R0, Tr


def read_calib(calib_path):
    """
    Parse a KITTI-style calib file written by save_kitti_calib:
      - Lines P0: …  P1: …  P2: …  P3: …
      - R0_rect:
      - Tr_velo_to_cam, Tr_velo_to_cam_0, Tr_velo_to_cam_1, Tr_velo_to_cam_2, Tr_velo_to_cam_3
    Returns:
      P: dict[int, np.ndarray]  -> projection matrices
      R0: np.ndarray (3x3)      -> rectification matrix
      Tr_dict: dict[str, np.ndarray] -> all Tr_velo_to_cam* entries
    """
    P = {}
    R0 = None
    Tr_dict = {}

    with open(calib_path, 'r') as f:
        for line in f:
            toks = line.strip().split()
            key = toks[0].rstrip(':')  # e.g., 'P0', 'R0_rect', 'Tr_velo_to_cam_1'
            vals = list(map(float, toks[1:]))

            if key.startswith('P'):
                cam_id = int(key[1])
                P[cam_id] = np.array(vals).reshape(3, 4)

            elif key == 'R0_rect':
                R0 = np.array(vals).reshape(3, 3)

            elif key.startswith('Tr_velo_to_cam'):
                Tr_dict[key] = np.array(vals).reshape(3, 4)

    return P, R0, Tr_dict

def read_label(label_path):
    """
    Parse KITTI label_2 format :
    class, truncated, occluded, alpha, x1,y1,x2,y2, h,l,w, x,y,z, ry
    """
    boxes = []
    with open(label_path, 'r') as f:
        for line in f:
            toks = line.strip().split()
            cls = toks[0]
            # skip toks[1:4]
            x1,y1,x2,y2 = map(float, toks[4:8])
            h,l_,w = map(float, toks[8:11])
            x,y,z = map(float, toks[11:14])
            ry = float(toks[14])
            boxes.append(dict(dim=(h,l_,w), loc=(x,y,z), ry=ry))
    return boxes

def compute_box_corners(h, l, w, loc, ry):
    """
    Reconstruct the 8 corners of a KITTI box in camera coords.
      - loc is the center of the bottom face.
      - dims are h,height; l,length (forward); w,width (right).
      - ry is yaw around camera Y.
    """
    x0,y0,z0 = loc
    # bottom corners (order: front‐right, back‐right, back‐left, front‐left)
    x_c = [ w/2,  w/2, -w/2, -w/2]
    y_c = [   0,    0,    0,    0 ]
    z_c = [ l/2, -l/2, -l/2,  l/2]
    R = np.array([
        [ np.cos(ry), 0, np.sin(ry)],
        [          0, 1,          0],
        [-np.sin(ry), 0, np.cos(ry)]
    ])
    corners = []
    for xc,yc,zc in zip(x_c,y_c,z_c):
        pt = R @ np.array([xc,yc,zc]) + np.array([x0,y0,z0])
        corners.append(pt)
    # top corners
    for i in range(4):
        pt = corners[i] + np.array([0, -h, 0])
        corners.append(pt)
    return np.array(corners)  # (8,3)

# ——— Visualization routines ———
import open3d as o3d
import numpy as np
import os

def visualize_lidar_with_boxes(kitti_dir, frame):
    """Open3D view: raw LiDAR + 3D boxes in vehicle coords."""
    # load
    pts = load_velodyne_bin(os.path.join(kitti_dir,'velodyne', frame+'.bin'))
    P, R0, Tr = read_calib(os.path.join(kitti_dir,'calib', frame+'.txt'))
    boxes = read_label(os.path.join(kitti_dir,'label_2', frame+'.txt'))

    # invert trimatrix & rect
    R_cam = Tr['Tr_velo_to_cam'][:, :3]
    t_cam = Tr['Tr_velo_to_cam'][:, 3]
    R0_inv = np.linalg.inv(R0) if R0 is not None else np.eye(3)

    # build line‐sets
    line_sets = []
    for b in boxes:
        # corners in cam
        cor_cam = compute_box_corners(*b['dim'], b['loc'], b['ry'])
        # undo rect
        cor_cam = (R0_inv @ cor_cam.T).T
        # cam→velo: x_velo = R_cam.T ( x_cam − t_cam )
        cor_velo = (R_cam.T @ (cor_cam - t_cam).T).T

        ls = o3d.geometry.LineSet()
        ls.points = o3d.utility.Vector3dVector(cor_velo)
        edges = [
            [0,1],[1,2],[2,3],[3,0],
            [4,5],[5,6],[6,7],[7,4],
            [0,4],[1,5],[2,6],[3,7]
        ]
        ls.lines = o3d.utility.Vector2iVector(edges)
        ls.colors = o3d.utility.Vector3dVector([[1,0,0] for _ in edges])
        line_sets.append(ls)

    # draw
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(pts)
    o3d.visualization.draw_geometries([pcd, *line_sets])

# def overlay_lidar_on_image(kitti_dir, frame, downsample=10):
#     """OpenCV window: project LiDAR → image and draw points."""

#     print(os.path.join(kitti_dir,'image_2', frame+'.png'))
#     img = cv2.imread(os.path.join(kitti_dir,'image_2', frame+'.png'))
#     pts = load_velodyne_bin(os.path.join(kitti_dir,'velodyne', frame+'.bin'))
#     P, R0, Tr = read_calib(os.path.join(kitti_dir,'calib', frame+'.txt'))

#     # intrinsics from P2
#     P2 = P[2]  # image_2 folder → P2
#     fx, fy = P2[0,0], P2[1,1]
#     cx, cy = P2[0,2], P2[1,2]
#     K = np.array([[fx,0,cx],[0,fy,cy],[0,0,1]])

#     # cam transf
#     R_cam = Tr[:, :3]; t_cam = Tr[:,3]
#     pts_cam = (R_cam @ pts.T + t_cam.reshape(3,1)).T
#     mask = pts_cam[:,2] > 0
#     pc = pts_cam[mask]

#     proj = (K @ pc.T).T
#     uv = np.stack([proj[:,0]/proj[:,2], proj[:,1]/proj[:,2]], axis=1).astype(int)

#     h,w = img.shape[:2]
#     valid = (uv[:,0]>=0)&(uv[:,0]<w)&(uv[:,1]>=0)&(uv[:,1]<h)
#     uv = uv[valid]

#     for p in uv[::downsample]:
#         cv2.circle(img, (int(p[0]), int(p[1])), 1, (0,255,0), -1)

#     img = cv2.resize(img, (img.shape[1]//2, img.shape[0]//2))

#     cv2.imshow('LiDAR-image', img)
#     cv2.waitKey(0)
#     cv2.destroyAllWindows()

def overlay_lidar_on_image(kitti_dir, frame, downsample=10):
    """OpenCV window: project LiDAR → image and draw points."""

    print(os.path.join(kitti_dir,'image_2', frame+'.png'))
    img = cv2.imread(os.path.join(kitti_dir,'image_2', frame+'.png'))
    pts = load_velodyne_bin(os.path.join(kitti_dir,'velodyne', frame+'.bin'))
    P, R0, Tr = read_calib(os.path.join(kitti_dir,'calib', frame+'.txt'))

    # intrinsics from P2
    P2 = P[2]  # image_2 folder → P2
    fx, fy = P2[0,0], P2[1,1]
    cx, cy = P2[0,2], P2[1,2]
    K = np.array([[fx,0,cx],[0,fy,cy],[0,0,1]])

    # cam transf
    R_cam = Tr['Tr_velo_to_cam'][:, :3]; t_cam = Tr['Tr_velo_to_cam'][:,3]
    pts_cam = (R_cam @ pts.T + t_cam.reshape(3,1)).T
    mask = pts_cam[:,2] > 0
    pc = pts_cam[mask]

    proj = (K @ pc.T).T
    uv = np.stack([proj[:,0]/proj[:,2], proj[:,1]/proj[:,2]], axis=1).astype(int)

    h,w = img.shape[:2]
    valid = (uv[:,0]>=0)&(uv[:,0]<w)&(uv[:,1]>=0)&(uv[:,1]<h)
    uv = uv[valid]

    for p in uv[::downsample]:
        cv2.circle(img, (int(p[0]), int(p[1])), 1, (0,255,0), -1)

    img = cv2.resize(img, (img.shape[1]//2, img.shape[0]//2))

    cv2.imshow('LiDAR-image', img)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

# def draw_boxes_on_image(kitti_dir, frame):
#     """OpenCV window: project 3D boxes → image and draw edges."""
#     img = cv2.imread(os.path.join(kitti_dir,'image_2', frame+'.png'))
#     P, R0, Tr = read_calib(os.path.join(kitti_dir,'calib', frame+'.txt'))
#     boxes = read_label(os.path.join(kitti_dir,'label_2', frame+'.txt'))

#     print(img.shape)

#     # intrinsics
#     P2 = P[2]
#     fx, fy = P2[0,0], P2[1,1]
#     cx, cy = P2[0,2], P2[1,2]
#     K = np.array([[fx,0,cx],[0,fy,cy],[0,0,1]])

#     # rectify
#     R0_inv = np.linalg.inv(R0) if R0 is not None else np.eye(3)

#     for b in boxes:
#         cor = compute_box_corners(*b['dim'], b['loc'], b['ry'])
#         # print(cor.shape)
#         #cor[:,0] += 0.15
#         # cam→rect undone
#         cor = (R0_inv @ cor.T).T
#         proj = (K @ cor.T).T
#         uv = np.stack([proj[:,0]/proj[:,2], proj[:,1]/proj[:,2]], axis=1).astype(int)
#         edges = [
#             [0,1],[1,2],[2,3],[3,0],
#             [4,5],[5,6],[6,7],[7,4],
#             [0,4],[1,5],[2,6],[3,7]
#         ]

#         for e in edges:
#             p1, p2 = tuple(uv[e[0]]), tuple(uv[e[1]])
#             cv2.line(img, p1, p2, (0,0,255), 2)

#     img = cv2.resize(img, (img.shape[1]//2, img.shape[0]//2))

#     cv2.imshow('Boxes-image', img)
#     cv2.imwrite('boxes_image.png', img)
#     cv2.waitKey(0)
#     cv2.destroyAllWindows()


# def draw_boxes_on_image(kitti_dir, frame):
#     """OpenCV window: project 3D boxes → image and draw edges."""
#     img = cv2.imread(os.path.join(kitti_dir,'image_2', frame+'.png'))
#     P, R0, Tr = read_calib(os.path.join(kitti_dir,'calib', frame+'.txt'))
#     boxes = read_label(os.path.join(kitti_dir,'label_2', frame+'.txt'))

#     print(img.shape)

#     # intrinsics
#     P2 = P[2]
#     fx, fy = P2[0,0], P2[1,1]
#     cx, cy = P2[0,2], P2[1,2]
#     K = np.array([[fx,0,cx],[0,fy,cy],[0,0,1]])

#     # rectify
#     R0_inv = np.linalg.inv(R0) if R0 is not None else np.eye(3)

#     for b in boxes:
#         cor = compute_box_corners(*b['dim'], b['loc'], b['ry'])
#         # print(cor.shape)
#         #cor[:,0] += 0.15
#         # cam→rect undone
#         cor = (R0_inv @ cor.T).T
#         proj = (K @ cor.T).T
#         uv = np.stack([proj[:,0]/proj[:,2], proj[:,1]/proj[:,2]], axis=1).astype(int)
#         edges = [
#             [0,1],[1,2],[2,3],[3,0],
#             [4,5],[5,6],[6,7],[7,4],
#             [0,4],[1,5],[2,6],[3,7]
#         ]

#         for e in edges:
#             p1, p2 = tuple(uv[e[0]]), tuple(uv[e[1]])
#             cv2.line(img, p1, p2, (0,0,255), 2)

#     img = cv2.resize(img, (img.shape[1]//2, img.shape[0]//2))

#     cv2.imshow('Boxes-image', img)
#     cv2.imwrite('boxes_image.png', img)
#     cv2.waitKey(0)
#     cv2.destroyAllWindows()
import os, cv2, numpy as np

def _to4x4(M34):
    T = np.eye(4, dtype=np.float64)
    T[:3, :4] = M34.reshape(3, 4)
    return T

def draw_boxes_on_image(kitti_dir, frame, cam_id=2):
    """
    Project 3D boxes (defined in cam2 coords, KITTI style) onto image_{cam_id}.
    Requires calib file to contain:
      - P0..P3
      - R0_rect
      - Tr_velo_to_cam          (for cam2)
      - Tr_velo_to_cam_{i}      (for other cameras you want to use)
    """
    # image for the chosen camera
    img = cv2.imread(os.path.join(kitti_dir, f'image_{cam_id}', frame + '.png'))
    assert img is not None, f"Image not found: image_{cam_id}/{frame}.png"

    P, R0, Tr_dict = read_calib(os.path.join(kitti_dir, 'calib', frame + '.txt'))
    boxes = read_label(os.path.join(kitti_dir, 'label_2', frame + '.txt'))

    # intrinsics for this camera
    Pi = P[cam_id]
    fx, fy, cx, cy = Pi[0,0], Pi[1,1], Pi[0,2], Pi[1,2]
    K = np.array([[fx, 0,  cx],
                  [0,  fy, cy],
                  [0,  0,  1 ]], dtype=np.float64)

    # Build cam2 -> cam_id transform using LiDAR as a bridge:
    #   T_cam2->cam_id = (T_cam_id<-velo) @ inv(T_cam2<-velo)
    # For cam2 itself, this is identity.
    if cam_id == 2:
        T_cam2_to_cami = np.eye(4, dtype=np.float64)
    else:
        # get T_cam_id<-velo
        key_i = f"Tr_velo_to_cam_{cam_id}"
        if key_i not in Tr_dict:
            raise ValueError(f"Missing {key_i} in calib; needed for cam_id={cam_id}")
        T_cami_velo = _to4x4(Tr_dict[key_i])

        # get T_cam2<-velo (canonical key or explicit _2)
        if "Tr_velo_to_cam" in Tr_dict:
            T_cam2_velo = _to4x4(Tr_dict["Tr_velo_to_cam"])
        elif "Tr_velo_to_cam_2" in Tr_dict:
            T_cam2_velo = _to4x4(Tr_dict["Tr_velo_to_cam_2"])
        else:
            raise ValueError("Missing Tr_velo_to_cam (or _2) in calib; needed as cam2 reference.")

        T_cam2_to_cami = T_cami_velo @ np.linalg.inv(T_cam2_velo)

    # (Most of your files have R0_rect = I; keep the same variable for compatibility.)
    R0_inv = np.linalg.inv(R0) if (R0 is not None) else np.eye(3, dtype=np.float64)

    # draw each box
    edges = [
        (0,1),(1,2),(2,3),(3,0),
        (4,5),(5,6),(6,7),(7,4),
        (0,4),(1,5),(2,6),(3,7)
    ]

    for b in boxes:
        # corners in cam2 coordinates (KITTI label convention)
        corners_cam2 = compute_box_corners(*b['dim'], b['loc'], b['ry'])  # (8,3)

        # cam2 -> cam_id
        corners_cam2_h = np.hstack([corners_cam2, np.ones((8,1))])        # (8,4)
        corners_cami_h = (T_cam2_to_cami @ corners_cam2_h.T).T            # (8,4)
        corners_cami   = corners_cami_h[:, :3]                             # (8,3)

        # (Optional) "undo rect" as your original code did; with R0=I this is a no-op.
        corners_cami = (R0_inv @ corners_cami.T).T

        # pinhole projection
        proj = (K @ corners_cami.T).T
        z = proj[:, 2:3]
        # keep only points in front of camera
        if np.any(z <= 1e-6):  # avoid divide-by-zero
            continue
        uv = proj[:, :2] / z
        uv = uv.astype(int)

        for e in edges:
            p1, p2 = tuple(uv[e[0]]), tuple(uv[e[1]])
            cv2.line(img, p1, p2, (0,0,255), 2)

    out = cv2.resize(img, (img.shape[1]//2, img.shape[0]//2))
    cv2.imshow(f'Boxes-camera{cam_id}', out)
    cv2.imwrite(f'boxes_image_cam{cam_id}.png', out)
    cv2.waitKey(0)
    cv2.destroyAllWindows()



Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:
import os
# ——— Example usage ———

if __name__ == "__main__":
    # kitti_folder = "data_to_nishad/KITTI_dataset_v13_mini_v0"       # adjust as needed
    kitti_folder = "example_hand_vs_auto"       # adjust as needed
    lidar_files = sorted(os.listdir(os.path.join(kitti_folder,"velodyne")))
    frames = [os.path.splitext(f)[0] for f in lidar_files if f.endswith(".bin")]
    frames.sort()
    for idx in range(0,2):#range(len(frames)):
        frame = frames[idx]
        print("visualizing frame:", frame)

        visualize_lidar_with_boxes(kitti_folder, frame) #Press cross to close the window
        overlay_lidar_on_image(kitti_folder, frame) #Press any key to close the window
        #Overlay done
        draw_boxes_on_image(kitti_folder, frame) #Press any key to close the window


visualizing frame: 004287
example_hand_vs_auto/image_2/004287.png
visualizing frame: 016893
example_hand_vs_auto/image_2/016893.png
